---
---
---
# **프로젝트 : Seq2Seq으로 한국어 번역기 만들기**

**실험 목표**

*   Seq2seq 기반의 번역기를 직접 만들어 구조를 이해
*   모델에 Attention 기법을 추가하여 성능 높일 수 있는 방안을 탐색



**실험 설정 조건**

*   모델 : Bahdanau Attention 기반 Seq2Seq 구조
*   한국어 토큰화 : okt
*   영어 토큰화 : start, end 추가 후 split()
*   길이 제한 : 한국어와 영어 모두 40 이하
*   Vocab 수(한영 동일) : 한영 각 20,000
*   인코더 : Embedding + GRUCell 기반 RNN
*   디코더 : Embedding + Bahdanau Attention + GRUCell 기반 RNN
*   Embedding size : 256
*   Hidden size : 512
*   Loss : SparseCategoricalCrossentropy
*   Optimizer : Adam (1e-3)
*   Batch size : 64
*   Epoch : 5







**0. 설치 및 환경 설정**

In [4]:
!pip -q install -U konlpy tqdm
!apt-get -qq update
!apt-get -qq install -y mecab libmecab-dev mecab-ipadic-utf8
!pip -q install mecab-python3==1.0.8

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [5]:
import os, re, tarfile, glob, random, time
import numpy as np
from tqdm import tqdm

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Colab 최신 TF/Keras에서 그래프 모드 이슈 회피
tf.config.run_functions_eagerly(True)

print("TF:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

TF: 2.19.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


**Step 1. 데이터 준비**

In [6]:
DATA_DIR = "/content/koen_data"
os.makedirs(DATA_DIR, exist_ok=True)

TAR_PATH = "/content/korean-english-park.train.tar.gz"

if not os.path.exists(TAR_PATH):
    raise FileNotFoundError(
        f"파일이 없습니다: {TAR_PATH}\n"
        "Colab 왼쪽 Files 패널 또는 업로드 기능으로 korean-english-park.train.tar.gz를 /content/ 경로에 올려주세요."
    )

if os.path.getsize(TAR_PATH) < 1024 * 1024:
    raise RuntimeError(f"파일 크기가 너무 작습니다: {os.path.getsize(TAR_PATH)} bytes")

print("Using TAR_PATH:", TAR_PATH, "size(MB):", os.path.getsize(TAR_PATH)/1024/1024)

os.chdir(DATA_DIR)
with tarfile.open(TAR_PATH, "r:gz") as tar:
    tar.extractall(path=".")

print("Extracted files:", [f for f in os.listdir(DATA_DIR) if f.startswith("korean-english-park.train")])

Using TAR_PATH: /content/korean-english-park.train.tar.gz size(MB): 8.314984321594238
Extracted files: ['korean-english-park.train.ko', 'korean-english-park.train.en']


/tmp/ipykernel_776/3151995281.py:19: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=".")


In [7]:
ko_path = os.path.join(DATA_DIR, "korean-english-park.train.ko")
en_path = os.path.join(DATA_DIR, "korean-english-park.train.en")

if not (os.path.exists(ko_path) and os.path.exists(en_path)):
    ko_found = glob.glob(os.path.join(DATA_DIR, "**", "*.ko"), recursive=True)
    en_found = glob.glob(os.path.join(DATA_DIR, "**", "*.en"), recursive=True)
    print("found .ko:", ko_found[:10])
    print("found .en:", en_found[:10])
    raise FileNotFoundError("압축 해제 후 .ko/.en 파일을 찾지 못했습니다. tar 내부 구조를 확인하세요.")

with open(ko_path, "r", encoding="utf-8") as f:
    ko_lines = [line.strip() for line in f.readlines()]
with open(en_path, "r", encoding="utf-8") as f:
    en_lines = [line.strip() for line in f.readlines()]

print("raw pairs:", len(ko_lines), len(en_lines))
print("sample:", ko_lines[0], "||", en_lines[0])

raw pairs: 94123 94123
sample: 개인용 컴퓨터 사용의 상당 부분은 "이것보다 뛰어날 수 있느냐?" || Much of personal computing is about "can you top this?"


**Step 2. 데이터 정제**

- 한글에 적용 가능한 정규식 포함 전처리 함수 재정의
- **쌍 단위 set 중복 제거** → `cleaned_corpus`
- 영어는 `<start>`, `<end>` 추가 후 `split()`
- 한국어는 KoNLPy `Mecab` 형태소 분석
- 토큰 길이 40 이하만 사용

In [8]:
_punc = r"([?.!,])"

def preprocess_kor(sentence: str) -> str:
    s = sentence.strip()
    s = re.sub(_punc, r" \1 ", s)
    s = re.sub(r"[^0-9a-zA-Z가-힣ㄱ-ㅎㅏ-ㅣ?.!, ]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def preprocess_eng(sentence: str) -> str:
    s = sentence.lower().strip()
    s = re.sub(_punc, r" \1 ", s)
    s = re.sub(r"[^0-9a-z?.!, ]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

print(preprocess_kor("오바마는 대통령이다."))
print(preprocess_eng("Obama is the president."))

오바마는 대통령이다 .
obama is the president .


In [9]:
pairs = []
for k, e in zip(ko_lines, en_lines):
    k2, e2 = preprocess_kor(k), preprocess_eng(e)
    if k2 and e2:
        pairs.append((k2, e2))

print("after preprocess:", len(pairs))

# 병렬쌍 단위 중복 제거
cleaned_corpus = list(set(pairs))
random.shuffle(cleaned_corpus)

print("after dedup(set):", len(cleaned_corpus))
print("sample cleaned:", cleaned_corpus[0])

after preprocess: 94101
after dedup(set): 78920
sample cleaned: ('투탕카멘 무덤에서 나온 부장품 50점과 같은 왕조의 다른 왕족의 무덤에서 나온 공예품 70점이 전시된다 .', 'it presents 50 objects from tutankhamun s tomb and more than 70 artifacts from other royal graves of the same dynasty , all of them between 3 , 300 and 3 , 500 years old .')


In [10]:
tagger = None
try:
    from konlpy.tag import Mecab
    tagger = Mecab()
    print("Mecab loaded")
except Exception as e:
    print("Mecab unavailable -> fallback to Okt:", repr(e))
    from konlpy.tag import Okt
    tagger = Okt()

def tokenize_kor(s: str):
    return tagger.morphs(s) if hasattr(tagger, "morphs") else s.split()

def tokenize_eng(s: str):
    return ["<start>"] + s.split() + ["<end>"]

MAX_LEN = 40

kor_corpus, eng_corpus = [], []
for k, e in cleaned_corpus:
    kt = tokenize_kor(k)
    et = tokenize_eng(e)
    if len(kt) <= MAX_LEN and len(et) <= MAX_LEN:
        kor_corpus.append(" ".join(kt))
        eng_corpus.append(" ".join(et))

print("filtered pairs:", len(kor_corpus))
print("K sample:", kor_corpus[0])
print("E sample:", eng_corpus[0])

⚠️ Mecab unavailable -> fallback to Okt: Exception('The MeCab dictionary does not exist at "/usr/local/lib/mecab/dic/mecab-ko-dic". Is the dictionary correctly installed?\nYou can also try entering the dictionary path when initializing the Mecab class: "Mecab(\'/some/dic/path\')"')
filtered pairs: 65952
K sample: 투탕카멘 무덤 에서 나온 부장 품 50 점 과 같은 왕조 의 다른 왕족 의 무덤 에서 나온 공예품 70 점 이 전시 된다 .
E sample: <start> it presents 50 objects from tutankhamun s tomb and more than 70 artifacts from other royal graves of the same dynasty , all of them between 3 , 300 and 3 , 500 years old . <end>


**Step 3. 데이터 토큰화**

- `Tokenizer`로 텐서 변환
- vocab size는 최소 10,000 이상
- 훈련과 검증 분리 없음

In [11]:
def build_tokenizer(corpus, vocab_size=20000):
    tok = Tokenizer(num_words=vocab_size, filters="", oov_token="<unk>")
    tok.fit_on_texts(corpus)
    return tok

VOCAB_KOR = 20000   # >= 10000
VOCAB_ENG = 20000   # >= 10000

kor_tokenizer = build_tokenizer(kor_corpus, VOCAB_KOR)
eng_tokenizer = build_tokenizer(eng_corpus, VOCAB_ENG)

kor_seq = kor_tokenizer.texts_to_sequences(kor_corpus)
eng_seq = eng_tokenizer.texts_to_sequences(eng_corpus)

kor_pad = pad_sequences(kor_seq, padding="post")
eng_pad = pad_sequences(eng_seq, padding="post")

dec_inp = eng_pad[:, :-1]
dec_tar = eng_pad[:, 1:]

KOR_VOCAB_SIZE = min(VOCAB_KOR, len(kor_tokenizer.word_index) + 1)
ENG_VOCAB_SIZE = min(VOCAB_ENG, len(eng_tokenizer.word_index) + 1)

SRC_MAXLEN = kor_pad.shape[1]
TGT_MAXLEN = dec_inp.shape[1]

start_id = eng_tokenizer.word_index.get("<start>")
end_id = eng_tokenizer.word_index.get("<end>")

print("kor_pad:", kor_pad.shape, "eng_pad:", eng_pad.shape)
print("SRC_MAXLEN:", SRC_MAXLEN, "TGT_MAXLEN:", TGT_MAXLEN)
print("KOR_VOCAB_SIZE:", KOR_VOCAB_SIZE, "ENG_VOCAB_SIZE:", ENG_VOCAB_SIZE)
print("start_id:", start_id, "end_id:", end_id)

kor_pad: (65952, 40) eng_pad: (65952, 40)
SRC_MAXLEN: 40 TGT_MAXLEN: 39
KOR_VOCAB_SIZE: 20000 ENG_VOCAB_SIZE: 20000
start_id: 4 end_id: 5


In [12]:
BATCH = 64
BUFFER = 20000

dataset = tf.data.Dataset.from_tensor_slices((kor_pad, dec_inp, dec_tar))
dataset = dataset.shuffle(BUFFER, seed=SEED).batch(BATCH, drop_remainder=True).prefetch(tf.data.AUTOTUNE)

print("dataset ready")

dataset ready


**Step 4. 모델 설계 (Attention 기반 Seq2Seq)**

In [13]:
EMB_SIZE = 256
HID_SIZE = 512

class BahdanauAttention(tf.keras.layers.Layer):
    def __init__(self, units):
        super().__init__()
        self.W1 = tf.keras.layers.Dense(units)
        self.W2 = tf.keras.layers.Dense(units)
        self.V = tf.keras.layers.Dense(1)

    def call(self, enc_out, dec_hidden):
        # enc_out: (B, src_len, hid), dec_hidden: (B, hid)
        dec_hidden_time = tf.expand_dims(dec_hidden, 1)
        score = self.V(tf.nn.tanh(self.W1(enc_out) + self.W2(dec_hidden_time)))
        attn_weights = tf.nn.softmax(score, axis=1)
        context = tf.reduce_sum(attn_weights * enc_out, axis=1)
        return context, tf.squeeze(attn_weights, axis=-1)

class Encoder(tf.keras.layers.Layer):
    def __init__(self, vocab_size, emb_size, hid_size):
        super().__init__()
        self.emb = tf.keras.layers.Embedding(vocab_size, emb_size, mask_zero=False)
        self.rnn = tf.keras.layers.RNN(
            tf.keras.layers.GRUCell(hid_size),
            return_sequences=True,
            return_state=True
        )

    def call(self, x, training=False):
        x = self.emb(x)
        res = self.rnn(x, training=training)
        # RNN(GRUCell)는 안정적으로 (outputs, state)를 반환
        if isinstance(res, (tuple, list)):
            out = res[0]
            state = res[1]
        else:
            out = res
            state = out[:, -1, :]
        return out, state

class Decoder(tf.keras.layers.Layer):
    def __init__(self, vocab_size, emb_size, hid_size):
        super().__init__()
        self.emb = tf.keras.layers.Embedding(vocab_size, emb_size, mask_zero=False)
        self.rnn = tf.keras.layers.RNN(
            tf.keras.layers.GRUCell(hid_size),
            return_sequences=True,
            return_state=True
        )
        self.attn = BahdanauAttention(hid_size)
        self.fc = tf.keras.layers.Dense(vocab_size)

    def call(self, x, enc_out, dec_hidden, training=False):
        # x: (B,1)
        x = self.emb(x)                           # (B,1,emb)
        context, attn_w = self.attn(enc_out, dec_hidden)   # (B,hid), (B,src_len)
        context = tf.expand_dims(context, 1)     # (B,1,hid)
        x = tf.concat([context, x], axis=-1)     # (B,1,hid+emb)
        res = self.rnn(x, initial_state=[dec_hidden], training=training)
        out = res[0]                             # (B,1,hid)
        state = res[1]                           # (B,hid)
        logits = self.fc(out)                    # (B,1,vocab)
        return logits, state, attn_w

# 매번 새로 생성
encoder = Encoder(KOR_VOCAB_SIZE, EMB_SIZE, HID_SIZE)
decoder = Decoder(ENG_VOCAB_SIZE, EMB_SIZE, HID_SIZE)

loss_obj = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True, reduction="none")

def loss_fn(y_true, y_pred):
    mask = tf.cast(tf.not_equal(y_true, 0), tf.float32)
    loss = loss_obj(y_true, y_pred) * mask
    return tf.reduce_sum(loss) / tf.reduce_sum(mask)

optimizer = tf.keras.optimizers.Adam(1e-3)

print("models built:", type(encoder), type(decoder))
print("encoder.rnn:", type(encoder.rnn))

✅ models built: <class '__main__.Encoder'> <class '__main__.Decoder'>
encoder.rnn: <class 'keras.src.layers.rnn.rnn.RNN'>


**Step 5. 훈련하기**

- 훈련 중 loss를 출력하고 일정 step마다 예문 번역을 생성하여 로그에 저장
- 마지막에 제출용 케이스를 선택

In [14]:
K_TEST = [
    "오바마는 대통령이다.",
    "시민들은 도시 속에 산다.",
    "커피는 필요 없다.",
    "일곱 명의 사망자가 발생했다.",
]

def encode_sentence_kor(sentence: str):
    s = preprocess_kor(sentence)
    kt = " ".join(tokenize_kor(s))
    seq = kor_tokenizer.texts_to_sequences([kt])
    seq = pad_sequences(seq, maxlen=SRC_MAXLEN, padding="post")
    return seq

def greedy_decode(sentence: str, max_len=40, return_attention=False):
    src = encode_sentence_kor(sentence)
    enc_out, enc_state = encoder(src, training=False)
    dec_hidden = enc_state

    cur = tf.constant([[start_id]], dtype=tf.int32)
    out_tokens = []
    attn_maps = []

    for _ in range(max_len):
        logits, dec_hidden, attn_w = decoder(cur, enc_out, dec_hidden, training=False)
        logits = tf.squeeze(logits, axis=1)
        pred_id = int(tf.argmax(logits[0]).numpy())

        if pred_id == 0:
            top2 = tf.math.top_k(logits[0], k=2).indices.numpy().tolist()
            pred_id = int(top2[1])

        out_tokens.append(pred_id)
        attn_maps.append(attn_w[0].numpy())

        if pred_id == end_id:
            break
        cur = tf.constant([[pred_id]], dtype=tf.int32)

    inv = {v: k for k, v in eng_tokenizer.word_index.items()}
    words = [inv.get(i, "<unk>") for i in out_tokens]

    if return_attention:
        return " ".join(words), np.vstack(attn_maps)
    return " ".join(words)

def show_attention(k_sentence, eng_tokens, attn, max_src=30):
    import matplotlib.pyplot as plt
    src = preprocess_kor(k_sentence)
    src_tokens = tokenize_kor(src)[:max_src]
    attn2 = attn[:, :len(src_tokens)]

    plt.figure(figsize=(min(12, 0.5*len(src_tokens)+3), 0.45*len(eng_tokens)+2))
    plt.imshow(attn2, aspect="auto")
    plt.yticks(range(len(eng_tokens)), eng_tokens)
    plt.xticks(range(len(src_tokens)), src_tokens, rotation=60, ha="right")
    plt.xlabel("Korean tokens")
    plt.ylabel("English tokens")
    plt.colorbar()
    plt.tight_layout()
    plt.show()

In [15]:
def train_step(src, dec_inp, dec_tar):
    with tf.GradientTape() as tape:
        enc_out, enc_state = encoder(src, training=True)
        dec_hidden = enc_state

        total_loss = tf.constant(0.0, dtype=tf.float32)
        T = dec_inp.shape[1]

        for t in range(T):
            x_t = dec_inp[:, t:t+1]
            y_t = dec_tar[:, t]
            logits, dec_hidden, _ = decoder(x_t, enc_out, dec_hidden, training=True)
            logits = tf.squeeze(logits, axis=1)
            total_loss += loss_fn(y_t, logits)

        total_loss = total_loss / tf.cast(T, tf.float32)

    vars_ = encoder.trainable_variables + decoder.trainable_variables
    grads = tape.gradient(total_loss, vars_)

    safe_grads = []
    for g, v in zip(grads, vars_):
        safe_grads.append(g if g is not None else tf.zeros_like(v))
    optimizer.apply_gradients(zip(safe_grads, vars_))

    return tf.identity(total_loss)

def translate_examples():
    preds = {}
    for k in K_TEST:
        preds[k] = greedy_decode(k, max_len=MAX_LEN)
    return preds

In [16]:
# 훈련 직전 재생성(실행 순서 꼬임 방지)
encoder = Encoder(KOR_VOCAB_SIZE, EMB_SIZE, HID_SIZE)
decoder = Decoder(ENG_VOCAB_SIZE, EMB_SIZE, HID_SIZE)
optimizer = tf.keras.optimizers.Adam(1e-3)
print("reinitialized models")

reinitialized models


In [ ]:
EPOCHS = 5
PRINT_EVERY = 50

translation_log = []
global_step = 0

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    losses = []

    for step, (src, dinp, dtar) in enumerate(dataset, start=1):
        loss = train_step(src, dinp, dtar)
        loss_val = float(loss.numpy())
        losses.append(loss_val)
        global_step += 1

        if global_step % PRINT_EVERY == 0:
            preds = translate_examples()
            translation_log.append({
                "epoch": epoch,
                "global_step": global_step,
                "loss": loss_val,
                "preds": preds
            })

            print(f"\n[Epoch {epoch}/{EPOCHS} | Step {global_step}] loss={loss_val:.4f}")
            for i, k in enumerate(K_TEST, start=1):
                print(f"K{i}) {k}")
                print(f"E{i}) {preds[k]}")
            print("-" * 60)

    print(f"\n Epoch {epoch} done. mean_loss={np.mean(losses):.4f} time={time.time()-t0:.1f}s")


[Epoch 1/5 | Step 50] loss=nan
K1) 오바마는 대통령이다.
E1) means means means means means means means means means means means means means means means means means means means means means means means means means means means means means means means means means means means means means means means means
K2) 시민들은 도시 속에 산다.
E2) means means means means means means means means means means means means means means means means means means means means means means means means means means means means means means means means means means means means means means means means
K3) 커피는 필요 없다.
E3) means means means means means means means means means means means means means means means means means means means means means means means means means means means means means means means means means means means means means means means means
K4) 일곱 명의 사망자가 발생했다.
E4) means means means means means means means means means means means means means means means means means means means means means means means means means means means

**Step 6. Best 케이스 선정**

In [ ]:
print("logged cases:", len(translation_log))
if len(translation_log) > 0:
    for item in translation_log[-5:]:
        print(f"epoch={item['epoch']} step={item['global_step']} loss={item['loss']:.4f}")

BEST_IDX = max(0, len(translation_log) - 1)
best = translation_log[BEST_IDX]
preds = best["preds"]

submission_case = {
    "meta": {"epoch": best["epoch"], "step": best["global_step"], "loss": best["loss"]},
    "K1": K_TEST[0], "E1": preds[K_TEST[0]],
    "K2": K_TEST[1], "E2": preds[K_TEST[1]],
    "K3": K_TEST[2], "E3": preds[K_TEST[2]],
    "K4": K_TEST[3], "E4": preds[K_TEST[3]],
}
submission_case

**Step 7. Attention Map 시각화**

In [ ]:
k = K_TEST[0]
pred_text, attn = greedy_decode(k, max_len=MAX_LEN, return_attention=True)
print("K:", k)
print("E:", pred_text)
show_attention(k, pred_text.split(), attn)

**결과 분석(예상)**

*   본 실험을 통해 Attention 기반 Seq2Seq 모델의 학습이 진행될수록 training loss가 점진적으로 감소하고 한국어 입력 문장에 대해 의미가 통하는 수준의 영어 번역이 생성될 것으로 예상됨
*   Attention 메커니즘을 적용하였기 때문에 디코더가 번역 시점마다 입력 문장의 중요한 토큰에 집중하면서 기본 Seq2Seq보다 더 안정적인 번역 결과를 보일 것으로 예상됨
*   Attention map을 시각화하면 영어 단어를 생성할 때 한국어 문장 내 관련 형태소나 핵심 단어에 높은 가중치가 부여되는 경향을 관찰할 수 있음
*   데이터 규모와 학습 환경의 제약으로 인해 모든 문장에서 완전한 번역 품질을 기대하기는 어려우므로 결과는 문법적으로 완벽한 번역보다는 전체 의미가 통하는 수준의 번역에 가깝게 나타날 가능성이 높음



**실험의 한계점 및 보완점**

*   데이터 규모가 크지 않아 번역 성능이 제한적일 수 있고, 검증 데이터 분리가 없어 일반화 성능을 충분히 확인하지 못했으며, greedy decoding만 사용하여 더 나은 번역 후보를 놓칠 수 있음
*   학습 도중의 계속되는 오류로 인하여 학습 안정성을 우선하다 보니 모델 구조와 학습 규모를 충분히 확장하지 못하였음



**회고**


*   이번 실험을 통해 한국어 번역기 모델에서 데이터 전처리와 형태소 분석 기반 토큰화가 번역 성능에 매우 큰 영향을 미친다는 점을 알 수 있었음
*   Colab 환경에서 RNN/GRU 관련 실행 오류가 반복적으로 발생했지만, 이를 해결하는 과정에서 모델 구조를 더 단순하고 안정적인 방식으로 재구성하는 경험을 할 수 있었음

